# Sprint 3 · Manage and recover the selected model

Linden Facilities needs 24 hourly energy-demand predictions as one file within 60 seconds. Work locally in the same clone and Python 3.12 project environment. These notebook sections follow the Campus journey in order and import the same shared core modules used by the project.

Follow [repository setup](https://github.com/richhiey/ml-llm-ops/blob/main/docs/setup.md), select its environment as the notebook kernel, and run sections in order. Read the task and make your prediction before executing a probe. Cells tagged `student-work` are your independent attempts; finish those decisions before running a later cell that consumes them. Guided outputs alone do not complete an independent task.

Keep one set of work and reuse completed runs. A fresh kernel reopens `work/project.json`; it does not silently choose another model. The core plotting functions display actual evidence and save SVG/PNG copies under `work/figures`. Exported Campus figures are verified reference examples; your custom experiment can produce different observations.


In [ ]:
from IPython import get_ipython
get_ipython().run_line_magic("matplotlib", "inline")
from pathlib import Path
import sys, json
if sys.version_info[:2] != (3, 12):
    raise RuntimeError("Use the project's Python 3.12 environment from the local setup guide.")
from linden_mlops import (Config, Workspace, ingest, preprocess, train, evaluate,
    run_pipeline, export_runs, verify_summary, register, approve, activate,
    batch_predict, reproduce)
from linden_mlops.lifecycle import ROOT, MODEL_NAME, code_revision
from linden_mlops.project import Project
from linden_mlops.dev import course_cli, project_tests, project_python, prepare_case
from linden_mlops.course_evidence import prepare
from linden_mlops.visuals import (
    plot_contract, plot_experiments, plot_identity, plot_recovery, save_figure)
from IPython.display import display
import pandas as pd
import numpy as np
progress = Project()
project = PACKAGE = ROOT
state = progress.work
ws = Workspace(progress.tracking)
figures = state / 'figures'
print("Package source:", ROOT.name)
print("Saved decision fields:", sorted(progress.data))
print("Source identity:", code_revision())


## A-S3-01 · From "which file is production?" to a registry

A registered model groups versions. Each version points to one source run and artifact; tags hold validation evidence; the mutable `active` alias chooses what the next job resolves. Registering alone does not approve or deploy a model. Old staging/production/archived stage labels are superseded vocabulary: reason with aliases, tags and environment-specific registered models.

| Evidence | Question |
|---|---|
| Run -> model artifact | What execution produced it? |
| Version + validation tags | Which artifact was checked? |
| Active alias | What will a new job resolve? |
| Retained alias event | What changed, when and why? |

Continue with the exact saved validation choice, not a fresh alpha `1` baseline.

### Learning outcome

Explain 3 problems a model registry solves that ad hoc file-saving does not - versioning, alias/tag-based lifecycle control, and auditability - and recognise staging, production, and archived as superseded stage vocabulary replaced by aliases, tags, and environment-specific registered models.

[Campus explanation, task and post-attempt feedback](https://github.com/richhiey/ml-llm-ops/blob/main/docs/course/sprint-3/01-from-which-file-is-production-to-a-registry/lesson.md).


In [ ]:
selection = progress.selected()
print(selection)
selected_run = ws.client.get_run(selection['run_id'])
assert selected_run.data.params['evaluation_partition'] == 'validation'

## A-S3-02 · Registering and promoting a model

Run the first cell to register the selected artifact. The saved registry identity is reused on reopening. On its first execution, activation while pending must fail. This boundary is our local application's policy, not MLflow authorization. Approval does not itself execute tests.

### Learning outcome

Use the provided model registry scaffold to register a new model version, record its validation status with tags, and promote it by assigning the scaffold's active-model alias.

[Campus explanation, task and post-attempt feedback](https://github.com/richhiey/ml-llm-ops/blob/main/docs/course/sprint-3/02-registering-and-promoting-a-model/lesson.md).


In [ ]:
if not progress.data.get('registry'):
    version = register(ws, selection['run_id'])
    progress.save(registry={'version': version, 'run_id': selection['run_id']})
version = progress.require('registry')['version']
mv = ws.client.get_model_version(MODEL_NAME, version)
if mv.tags.get('validation_status') == 'pending':
    try:
        activate(ws, version, 'Deliberate pending probe')
    except ValueError as error:
        print('Expected rejection:', error)
    else:
        raise AssertionError('Pending version unexpectedly activated')
else:
    print('Reusing an already checked version:', version, mv.tags)
assert mv.run_id == selection['run_id']

Run the contract test against saved source, inspect the chosen run's finite validation RMSE (at most `5` kWh), then record an honest reason and activate. The batch resolves that version once. Compare its source run and version with registration; current aliases cannot prove what an earlier job loaded.

In [ ]:
if not progress.data.get('promotion'):
    project_tests(ROOT, 'tests/test_spine.py::test_contract_extension_and_unchanged_pipeline', '-q')
    approve(ws, version, 'Observed passing input contract; guarded finite validation RMSE <= 5 kWh')
    event = activate(ws, version, 'Selected Sprint 2 configuration after validation checks')
    output, inference = batch_predict(ws)
    progress.save(promotion={'event': event, 'inference': inference})
promotion = progress.require('promotion')
assert promotion['inference']['run_id'] == selection['run_id']
assert promotion['inference']['resolved_version'] == version
print(promotion)
print(ws.client.get_model_version(MODEL_NAME, version).tags)

### Follow the same artifact into the batch

The diagram is derived from recorded identities, not a picture of the registry UI. Registration identifies an artifact; approval records a checked policy decision; the next batch resolves a version once. A later alias lookup cannot establish what an earlier job loaded.


In [ ]:
registered = ws.client.get_model_version(MODEL_NAME, version)
identity_figure = plot_identity(selection['run_id'], version, registered.run_id,
                                promotion['inference'])
display(identity_figure)
print(identity_figure.linden_data)
identity_paths = save_figure(identity_figure, figures / 'model-identity__02__en')


## A-S3-03 · What full reproducibility requires

A metric match alone is insufficient. The manifest ties code content, data bytes, config, evaluation partition, environment and fitted model to one run. The lock defines dependency resolution; the recorded environment also identifies interpreter, installed packages and platform. Retaining only a pickle cannot recover the training recipe.

Inspect the manifest and source archive in MLflow. Which field would reveal a changed split or a different code edit? Which identifier is expected to change when you rerun?

### Learning outcome

Identify the code version, data snapshot, parameters/configuration, environment, and model artifact that must be traceable together for a training run to be reproducible.

[Campus explanation, task and post-attempt feedback](https://github.com/richhiey/ml-llm-ops/blob/main/docs/course/sprint-3/03-what-full-reproducibility-requires/lesson.md).


In [ ]:
recipe_path = Path(ws.client.download_artifacts(selection['run_id'], 'reproduction/manifest.json'))
recipe = json.loads(recipe_path.read_text())
print({key: recipe[key] for key in ['run_id', 'config', 'code_revision', 'data_sha256', 'evaluation_partition', 'model_uri']})
print('Environment:', recipe['environment'])

## A-S3-04 · Reproducing a past training run

Use the recorded selected run and the `reproduce` API/CLI to rerun into a new store, then inspect `reproduction-check.json`. Check source, data, config, partition and environment before interpreting the absolute RMSE delta. The declared tolerance is `1e-6` kWh. Explain changed IDs versus invariant recipe fields.

Terminal route: `uv run linden-mlops reproduce --destination work/past-replay`. This is a same-environment replay, not yet the new-source/new-environment proof. Keep one replay and reuse it.

### Learning outcome

Given a past experiment logged in MLflow, use its recorded code, data, configuration, environment, and registry lineage to reproduce the evaluation result within the tolerance declared in the clean-state verification specification.

[Campus explanation, task and post-attempt feedback](https://github.com/richhiey/ml-llm-ops/blob/main/docs/course/sprint-3/04-reproducing-a-past-training-run/lesson.md).


In [ ]:
# Your attempt: use reproduce(ws, selection['run_id'], state / 'past-replay')
# The destination must not exist. Inspect and explain the returned comparison.

## Check your work after attempting the task

<details>
<summary>Compare your result after the attempt</summary>

**Result you should get:** The replay has a new run ID, matching recipe fields and validation RMSE delta at most 1e-6 kWh. It reuses your source/environment; it is not the final clean receiving proof.

**How:** Inspect recoverable files before the score. If the score agrees but lineage differs, retain the failure and restore the recorded source/data/environment rather than relax the tolerance.

</details>


## A-S3-05 · Rolling back a bad model version

The main cumulative project's selection stays intact. Create an isolated incident below once. The client now requires validation RMSE at most `1.6` kWh; a trial is active. Read actual version tags and raw source runs, identify the prior approved version that qualifies, and reassign its alias with a reason. Preparation does not perform a rollback.

Use the API lookup or `activate --state work/rollback --version ... --reason ...`, then exercise a new batch. Tag the trial incident and retain both artifacts. Explain the old/new versions, source run, timestamp and reason in `audit.jsonl`; this mutable helper-mediated log is not a tamper-proof transaction.

### Learning outcome

Reassign the active-model alias from a bad version to the previous approved version, and document what the registry audit evidence captured during the rollback.

[Campus explanation, task and post-attempt feedback](https://github.com/richhiey/ml-llm-ops/blob/main/docs/course/sprint-3/05-rolling-back-a-bad-model-version/lesson.md).


In [ ]:
incident_state = state / 'rollback'
if not incident_state.exists():
    prepare_case('rollback', incident_state)
incident = Workspace(incident_state)
print((incident_state / 'case.json').read_text())
display(export_runs(incident))
for item in incident.client.search_model_versions(f"name='{MODEL_NAME}'"):
    print(item.version, item.run_id, item.tags, item.aliases)

In [ ]:
# Inspect before/after selection; choose the qualifying prior approved version.
# Activate it with an evidence-based reason, tag the trial incident, then batch_predict(incident).
# Compare the new job's run/version and retained audit event.

## Check your work after attempting the task

<details>
<summary>Compare your result after the attempt</summary>

**Result you should get:** The prior qualifying approved version becomes active; the trial remains inspectable and incident-tagged. The next batch records the restored version/source run.

**How:** Inspect the actual revised limit, raw metrics and tags before choosing. Explain old/new versions and audit limitations; a higher counter or an old batch cannot prove recovery.

</details>


## A-S3-06 · Registry lifecycle deliverable

Prepare Harbor's intake once. Its revised limit is `1.45` kWh, and the newest artifact is pending. Determine the qualifying prior approved version using its actual evidence, restore it, tag the trial incident and exercise the next batch. Recover that version's run into a new store and check lineage plus tolerance. Reuse the main project's registration/promotion as your connected lifecycle example.

Keep all three artifacts. Explain why the highest version number is not the answer by itself and what your audit can/cannot establish. No new pipeline or written submission is needed.

### Learning outcome

Deliverable: a model registered, tagged with validation evidence, assigned the active-model alias, then rolled back by alias reassignment - with a documented audit trail and the lineage needed to reproduce the approved run within the tolerance declared in the clean-state verification specification.

[Campus explanation, task and post-attempt feedback](https://github.com/richhiey/ml-llm-ops/blob/main/docs/course/sprint-3/06-registry-lifecycle-deliverable/lesson.md).


In [ ]:
harbor_state = state / 'harbor-registry'
if not harbor_state.exists():
    prepare_case('registry-assessment', harbor_state)
harbor = Workspace(harbor_state)
print((harbor_state / 'case.json').read_text())
display(export_runs(harbor))
for item in harbor.client.search_model_versions(f"name='{MODEL_NAME}'"):
    print(item.version, item.run_id, item.tags, item.aliases)

In [ ]:
# Resolve Harbor's incident using taught operations and explicit IDs.
# Check the restored job and reproduce its source run into a new state.
# Preserve the main cumulative project's selection.

## Check your work after attempting the task

<details>
<summary>Compare your result after the attempt</summary>

**Result you should get:** The restored version passes Harbor's revised 1.45-kWh rule and is approved. The newer pending artifact remains pending; all artifacts remain inspectable.

**How:** Match the registry to raw source runs, then verify the new batch and reproduce the restored run into a new state. Inspect the Campus self-check after your decision rather than assume a version counter.

</details>
